In [ ]:
!pip install wandb

In [ ]:
import wandb

wandb.login()

In [ ]:
import torch
from torch import nn
import torch.backends.cudnn as cudnn
import pandas as pd
from torch.utils.data import DataLoader, Dataset
import os
from PIL import Image
import numpy as np
import copy
from tqdm import tqdm

class FSRCNN(nn.Module):
    def __init__(self,
                scale_factor,
                num_channels,
                # number of filters in the first and last layers
                d=56,
                # number of filters in the shrinking and mapping layers
                s=12,
                # number of mapping layers
                m=4):

        super(FSRCNN, self).__init__()

        self.first_part = nn.Sequential(
            nn.Conv2d(num_channels,
                      d,
                      kernel_size=5,
                      padding=5//2),
            nn.PReLU(d)
        )

        self.mid_part = [
            nn.Conv2d(d, s, kernel_size=1),
            nn.PReLU(s)
        ]

        # For every mapping layer
        for _ in range(m):
            # Add a convolution layer followed by a PReLU activation function
            self.mid_part.extend([
                nn.Conv2d(s,
                          s,
                          kernel_size=3,
                          padding=3//2),
                nn.PReLU(s)])

        # Add one more convolution layer followed by a PReLU activation function
        self.mid_part.extend([
            nn.Conv2d(s,
                      d,
                      kernel_size=1),
            nn.PReLU(d)])

        # Convert the list of layers into a sequential module
        self.mid_part = nn.Sequential(*self.mid_part)


        self.last_part = nn.ConvTranspose2d(d,
                                            num_channels,
                                            kernel_size=9,
                                            stride=scale_factor,
                                            padding=9//2,
                                            output_padding=scale_factor-1)

    def forward(self, x):
        x = self.first_part(x)
        x = self.mid_part(x)
        x = self.last_part(x)
        return x

In [ ]:
class AverageMeter(object):
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

In [ ]:
# Training configuration

cudnn.benchmark = True

num_channels = 3 # 3 for RGB, 1 for single-channel e.g. Y channel in YCbCr
scale_factor = 4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
learning_rate = 1e-3
batch_size = 16
num_epochs = 20

# wandb configuration
wandb.init(project="FSRCNN-Face-SR", config={
    "learning_rate": learning_rate,
    "batch_size": batch_size,
    "num_epochs": num_epochs,
    "lr": learning_rate,
    "scale_factor": scale_factor,
    "num_channels": num_channels,
}, reinit="finish_previous")

# Log evaluation results every 100 batches
log_eval_interval = 100


In [ ]:

import torchvision.transforms as transforms

# Read CSV with splits
dataframe = pd.read_csv("/kaggle/input/celeba-dataset/list_eval_partition.csv")

# Each dataframe contains the image file names
# belonging to that split
train_df = dataframe[dataframe['partition'] == 0]
validation_df = dataframe[dataframe['partition'] == 1]
test_df = dataframe[dataframe['partition'] == 2]

# Function that transforms from Pillow RGB to PyTorch tensor and normalizes to [0, 1]
transform = transforms.Compose([
    transforms.ToTensor()
])

class CelebADataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, img_dir: str, transform_func=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform_func = transform_func

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_filename = self.dataframe.loc[idx, 'image_id']
        img_path = os.path.join(self.img_dir, img_filename)

        hr_image = Image.open(img_path).convert('RGB')

        hr_adjusted_width = (hr_image.width // scale_factor) * scale_factor
        hr_adjusted_height = (hr_image.height // scale_factor) * scale_factor
        hr_image = hr_image.resize((hr_adjusted_width, hr_adjusted_height), resample=Image.BICUBIC)

        # create LR from HR
        lr_image = hr_image.resize((hr_adjusted_width // scale_factor, hr_adjusted_height // scale_factor),
                                resample=Image.BICUBIC)
        
        # Use PyTorch transforms to convert to tensor and normalize
        hr_image = transform(hr_image)
        lr_image = transform(lr_image)

        return lr_image, hr_image

In [ ]:
# Add seed for reproducibility
torch.manual_seed(0)

model = FSRCNN(scale_factor=scale_factor, num_channels=num_channels).to(device)

wandb.watch(model, log="all", log_freq=100)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam([
    {'params': model.first_part.parameters()},
    {'params': model.mid_part.parameters()},
    {'params': model.last_part.parameters(), 'lr': learning_rate * 0.1}
],
lr=learning_rate)

img_dir = "/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba"
train_loader = DataLoader(CelebADataset(train_df, img_dir), batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(CelebADataset(validation_df, img_dir), batch_size=1, shuffle=False)

best_weights = copy.deepcopy(model.state_dict())
best_epoch = 0
best_psnr = 0.0

for epoch in range(num_epochs):
    # Enter training mode
    model.train()

    epoch_losses = AverageMeter()

    with tqdm(total=(len(train_loader) - len(train_loader) % batch_size),
              ncols=80) as t:
        t.set_description(f'Epoch {epoch + 1}/{num_epochs}')

        # Returns data in batches
        for data in train_loader:
            lr_images, hr_images = data
            lr_images = lr_images.to(device)
            hr_images = hr_images.to(device)

            # Forward pass
            sr_images = model(lr_images)

            loss = criterion(sr_images, hr_images)

            # Update the average loss for the epoch
            epoch_losses.update(loss.item(), len(lr_images))
            # Also log the loss to wandb
            # TODO: log epoch only at the end of each epoch
            wandb.log({
                "train_loss": loss.item(),
                "train_loss_epoch_avg": epoch_losses.avg,
                "epoch": epoch + 1,
                "lr": learning_rate,
            })

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Add the average loss to the tqdm progress bar
            t.set_postfix(loss=f"{epoch_losses.avg:.6f}")
            t.update(len(lr_images))

        # Save model

    torch.save(model.state_dict(), f"fsrcnn_epoch_{epoch + 1}.pth")

    # Enter evaluation mode
    model.eval()
    epoch_psnr = AverageMeter()

    for index, data in enumerate(validation_loader):
        lr_images, hr_images = data
        lr_images = lr_images.to(device)
        hr_images = hr_images.to(device)

        # Disable gradient calculation for evaluation
        with torch.no_grad():
            sr_images = model(lr_images).clamp(0.0, 1.0)

            # log result to wandb every log_eval_interval batches
            if index % log_eval_interval == 0:
                wandb.log({
                    "evaluation_results": [
                        wandb.Image(lr_images[0].cpu(), caption="Low Resolution"),
                        wandb.Image(sr_images[0].cpu(), caption="Reconstructed"),
                        wandb.Image(hr_images[0].cpu(), caption="High Resolution")
                    ]
                })

        

        mse_loss = criterion(sr_images, hr_images).item()
        psnr = 10 * np.log10(1 / mse_loss)
        epoch_psnr.update(psnr, len(lr_images))

        wandb.log({
            "val_psnr": epoch_psnr.avg,
            "epoch": epoch + 1
        })

    print(f"Validation PSNR: {epoch_psnr.avg:.2f} dB")

    if epoch_psnr.avg > best_psnr:
        best_psnr = epoch_psnr.avg
        best_epoch = epoch + 1
        best_weights = copy.deepcopy(model.state_dict())
        print(f"New best model found at epoch {best_epoch} with PSNR: {best_psnr:.2f} dB")
        torch.save(best_weights, "fsrcnn_best.pth")

# Finish wandb
wandb.finish()